[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahirtonlopes/AI-and-Robotics-Trends/blob/main/Notebooks/Demo3_Fraud_Detection.ipynb)

# Demo 3 - Demonstração - Aprendizagem Supervisionada

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEG)

### Algoritmo para classificação de transações em possíveis fraudes/não fraude.

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. - https://www.kaggle.com/pwnpen/payment

In [ ]:
# Import de nossas bibliotecas

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score)

In [ ]:
'''
# Upload de nosso arquivo.

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
'''

In [ ]:
# Upload de nosso arquivo. Ref. - https://www.kaggle.com/pwnpen/payment
# Baixando nossa base de dados direto do repositório da disciplina (https://github.com/ahirtonlopes/AI-and-Robotics-Trends/) para o Google Colab

!curl -O https://raw.githubusercontent.com/ahirtonlopes/AI-and-Robotics-Trends/main/Datasets/payment_fraud.csv

In [ ]:
# Lendo dados a partir de nosso arquivo .csv
df = pd.read_csv('payment_fraud.csv')

In [ ]:
# Amostragem de cabeçalho e cinco primeiras linhas de nosso dataset
df.sample(5)

In [ ]:
# Verificando linhas e partir do índice de nosso dataset
len(df.index)

In [ ]:
# Verificando colunas de nosso dataset
df.columns

In [ ]:
# Verificando itens únicos de nossa coluna 'paymentMethod' de nosso dataset
(df['paymentMethod'].unique())

In [ ]:
# Agrupando número de itens de nossa coluna 'numItems' de nosso dataset
df.groupby('numItems').size().reset_index()

In [ ]:
# Tratando nossos dados da coluna 'pamentMethod' para formato numérico - Ref. https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html
df_one_hot = pd.get_dummies(df, columns=['paymentMethod'])

In [ ]:
df_one_hot.sample(3)

In [ ]:
# Divisão em sets de treinamento/teste (Regra de Pareto - 80/20 - Ref. https://pt.wikipedia.org/wiki/Princ%C3%ADpio_de_Pareto)
X_train, X_test, y_train, y_test = train_test_split(df_one_hot.drop('label', axis=1), df_one_hot['label'], test_size=0.2, random_state=42)

In [ ]:
len(X_train.columns)

In [ ]:
# (max_iter=1000: com 20 iterações o otimizador costuma avisar que não convergiu)
# Construção de nosso modelo usando Regressão Logística (Ref. https://edisciplinas.usp.br/pluginfile.php/3769787/mod_resource/content/1/09_RegressaoLogistica.pdf)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)

# Predição em dados de teste
y_pred = clf.predict(X_test)

In [ ]:
# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

In [ ]:
# Utilizando Árvore de Decisão
clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

# Avaliação de Acurácia, Precisão e Recall de nosso modelo
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("acuracia", accuracy)
print("precisao", precision)
print("recall", recall)

## O resultado é perfeito. Isso é bom ou é suspeito?

Os dois modelos acertam 100%. Antes de comemorar, três verificações: **(1)** comparar com um modelo trivial; **(2)** olhar a matriz de confusão; **(3)** procurar uma coluna que entregue a resposta.

> Nota: nas versões anteriores deste notebook, `accuracy_score`, `precision_score` e `recall_score` eram chamadas com os argumentos invertidos (`y_pred` antes de `y_test`), o que **troca precision e recall**. Com 100% não aparecia; com um modelo imperfeito apareceria. A ordem correta é `(y_verdadeiro, y_previsto)`.

In [ ]:
# (1) Baseline trivial: um 'modelo' que sempre responde 'normal'
import numpy as np
from sklearn.metrics import confusion_matrix

y_base = np.zeros(len(y_test), dtype=int)
print(f"Fraudes na base: {int(df['label'].sum())} de {len(df)} ({df['label'].mean():.2%})")
print(f"Fraudes no teste: {int(y_test.sum())} de {len(y_test)}")
print(f"Acurácia do 'sempre normal': {accuracy_score(y_test, y_base):.4f}  |  recall: {recall_score(y_test, y_base):.2f}")

In [ ]:
# (2) Matriz de confusão da árvore (linhas = real, colunas = previsto)
print(pd.DataFrame(confusion_matrix(y_test, y_pred), index=['real: normal', 'real: fraude'], columns=['previsto: normal', 'previsto: fraude']))

In [ ]:
# (3) Existe uma coluna que entrega a resposta? Compare as colunas por classe.
print(df.groupby('label')[['accountAgeDays', 'numItems', 'localTime', 'paymentMethodAgeDays']].agg(['min', 'max']).T)

# Uma regra de uma única pergunta:
regra = (df['accountAgeDays'] <= 1.5).astype(int)
print('\nRegra: accountAgeDays <= 1.5')
print(' acertos:', (regra == df['label']).mean())

**Leitura:** toda fraude tem `accountAgeDays = 1`, e as transações normais têm pelo menos 2. Uma única regra separa tudo. A base é sintética e fácil demais; um fraudador real envelheceria a conta. **Resultado perfeito em segurança é sinal de alerta: procure vazamento ou atalho.**

---

## Ato 2: removendo a coluna que vaza

Vamos refazer tudo **sem** o `accountAgeDays` e ver o modelo enfrentar o problema de verdade.

Duas mudanças: além de tirar a coluna, usamos `stratify=y` no `train_test_split`. Como as fraudes são só ~1,4% da base, sem estratificação o sorteio pode deixar proporções diferentes de fraude no treino e no teste; o `stratify` mantém a mesma proporção nos dois.

In [ ]:
from sklearn.metrics import f1_score

def avaliar(nome, y_true, y_pred):
    vn, fp, fn, vp = confusion_matrix(y_true, y_pred).ravel()
    print(nome)
    print(f'  acurácia {accuracy_score(y_true, y_pred):.4f} | precisão {precision_score(y_true, y_pred, zero_division=0):.4f} | recall {recall_score(y_true, y_pred):.4f} | f1 {f1_score(y_true, y_pred, zero_division=0):.4f}')
    print(f'  fraudes pegas: {vp} de {vp + fn}   |   alarmes falsos: {fp}')
    print()

# Mesma base, sem a coluna que vazava
X2 = df_one_hot.drop(['label', 'accountAgeDays'], axis=1)
y2 = df_one_hot['label']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
print(f'Fraudes no treino: {y2_train.mean():.4%}  |  no teste: {y2_test.mean():.4%}')
print()

lr2 = LogisticRegression(max_iter=1000).fit(X2_train, y2_train)
dt2 = DecisionTreeClassifier(random_state=42, max_depth=5).fit(X2_train, y2_train)
avaliar('Regressão Logística (sem accountAgeDays)', y2_test, lr2.predict(X2_test))
avaliar('Árvore de Decisão (sem accountAgeDays)', y2_test, dt2.predict(X2_test))

### Leia com atenção o que acabou de acontecer

A acurácia continua ótima (cerca de **98,5%**) e o recall é **zero**.

Os dois modelos aprenderam a estratégia do 'modelo burro': responder 'não é fraude' para tudo. Como ~98,6% das transações são legítimas, isso rende 98,5% de acurácia, e **nenhuma das fraudes do teste foi detectada**. Se olhássemos só a acurácia, colocaríamos esse modelo em produção. Foi a **matriz de confusão** (verdadeiros positivos = 0) que denunciou. É por isso que acurácia não serve sozinha em base desbalanceada.

---

## Ato 3: dizendo ao modelo que fraude custa mais caro

Ignorar a classe rara é matematicamente o que minimiza o erro total, mas, para o negócio, deixar passar uma fraude custa muito mais do que incomodar um cliente legítimo. O `class_weight='balanced'` faz cada erro na classe rara pesar proporcionalmente mais no treino.

In [ ]:
lr3 = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X2_train, y2_train)
dt3 = DecisionTreeClassifier(random_state=42, max_depth=5, class_weight='balanced').fit(X2_train, y2_train)
avaliar('Regressão Logística (balanceada)', y2_test, lr3.predict(X2_test))
avaliar('Árvore de Decisão (balanceada)', y2_test, dt3.predict(X2_test))

### O trade-off entre precisão e recall, na prática

O modelo balanceado passa a **pegar quase todas as fraudes** (recall alto), em troca de **milhares de alarmes falsos** e de uma precisão muito baixa: de cada 100 transações marcadas como fraude, poucas são fraude de verdade. A acurácia despenca, e fica **pior** que a do modelo inútil do Ato 2.

| Modelo | Acurácia | Precisão | Recall |
|---|---|---|---|
| Ato 1 (com vazamento) | 1,000 | 1,000 | 1,000 |
| Ato 2 (sem vazamento) | ~0,985 | 0,000 | 0,000 |
| Ato 3 (balanceado) | bem menor | muito baixa | alto |

**Qual é o melhor modelo?** A pergunta não tem resposta técnica; tem resposta de negócio. Se uma fraude perdida custa um estorno alto e um alarme falso custa só uma verificação, o Ato 3 é melhor. Se cada alarme falso bloqueia um cliente bom, não. O caminho real fica no meio: **ajustar o limiar** (próxima seção), buscar variáveis com poder preditivo de verdade (as que sobraram aqui são fracas) e escolher a métrica pelo **custo de cada erro**.

## Limiar de decisão

O modelo devolve uma **probabilidade**. O corte padrão (0,5) é só convenção. Vamos mover o corte no modelo do Ato 3 (sem o atalho) e ver o trade-off.

In [ ]:
proba = lr3.predict_proba(X2_test)[:, 1]
print('limiar | precisão | recall | fraudes pegas | alarmes falsos')
for t in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95):
    p = (proba >= t).astype(int)
    vn, fp, fn, vp = confusion_matrix(y2_test, p, labels=[0, 1]).ravel()
    print(f' {t:5.2f} | {precision_score(y2_test, p, zero_division=0):8.3f} | {recall_score(y2_test, p):6.3f} | {vp:4d} de {vp + fn} | {fp}')

**Leitura:** subir o limiar reduz os alarmes falsos e também as fraudes pegas. Não há corte 'certo': ele depende do custo de cada erro e da capacidade de revisão da equipe (Aula 2, seção de métricas).

## Para praticar

1. **Custo:** suponha que cada fraude perdida custe 20 vezes mais que um cliente bloqueado por engano. Qual limiar da tabela acima você escolheria? E se o custo for 2 vezes?
2. **Separação:** compare `stratify=y` com e sem estratificação, e rode `cross_val_score(..., scoring='recall', cv=5)`. O resultado se mantém?
3. **Outro modelo:** troque a regressão logística por `RandomForestClassifier(class_weight='balanced')` e compare precisão e recall no Ato 3.
4. **Novas transações:** monte 3 transações novas (sem `accountAgeDays`) e veja a probabilidade de fraude que o modelo atribui (`lr3.predict_proba`).
5. **Custo:** escreva em uma frase qual erro é mais caro no seu caso do trabalho final.